In [2]:
###Load packages###
import pandas as pd
import os
import ast
from scipy import stats
from matplotlib import pyplot as plt
from scipy.stats import pearsonr
from scipy.stats import spearmanr
from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind
import numpy as np
import statsmodels.formula.api as smf

###Load cleaned dataset###

#Set file paths
topdir = '/Users/sm6511/Desktop/Prediction-Accomodation-Exp'
study = 'Study2.0'
cleandir = os.path.join(topdir, f'data/{study}/Cleaned')
outputdir = os.path.join(topdir, f'Analysis/{study}')
outputdirCombined = os.path.join(topdir, f'data/Combined')
outputdirCleaned = os.path.join(topdir, f'data/{study}/Cleaned')
os.makedirs(outputdir, exist_ok=True)
os.makedirs(outputdirCleaned, exist_ok=True)

#Read in cleaned data 
accomodate_path = os.path.join(cleandir, f'{study}Accommodate.csv')
predict_path   = os.path.join(cleandir, f'{study}Predict.csv')

df_accommodate = pd.read_csv(accomodate_path)
df_predict   = pd.read_csv(predict_path)

df_accommodate['task'] = 'accommodate'
df_predict['task']   = 'predict'


print("Accommodate columns:", df_accommodate.columns.tolist())
print("Predict columns:", df_predict.columns.tolist())


Accommodate columns: ['participant', 'free_texts', 'feedback', 'food_amount', 'trial_stop_time', 'testing_image_order', 'testing_responses', 'training_categories', 'training_tail', 'training_wing', 'training_color', 'testing_categories', 'conditionOrder', 'training_image_order', 'attention_check', 'relevant_dim_1', 'relevant_dim_2', 'irrelevant_dim', 'color_high', 'color_low', 'wing_high', 'wing_low', 'tail_high', 'tail_low', 'wing_discrete_slider.response', 'wing_direction_slider.response', 'wing_continuous_slider.response', 'color_discrete_slider.response', 'color_direction_slider.response', 'color_continuous_slider.response', 'tail_discrete_slider.response', 'tail_direction_slider.response', 'tail_continuous_slider.response', 'task']
Predict columns: ['participant', 'training_responses', 'food_amount', 'error', 'feedback', 'trial_stop_time', 'testing_image_order', 'testing_responses', 'training_categories', 'training_tail', 'training_wing', 'training_color', 'testing_categories', 'c

In [3]:
#Converting string representations of lists back to lists

def parse_list_column(x):
    """take column entries that are strings representing lists and convert them to actual lists"""
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if x.startswith('[') and x.endswith(']'):
            return ast.literal_eval(x)
        else:
            return [x]
    return []
for col in ['training_tail', 'training_wing', 'training_color', 'training_image_order', 'training_categories', 'testing_categories']:
    df_accommodate[col] = df_accommodate[col].apply(parse_list_column)
    df_predict[col]   = df_predict[col].apply(parse_list_column)

df_accommodate['testing_responses'] = df_accommodate['testing_responses'].apply(ast.literal_eval)
df_accommodate['food_amount'] = df_accommodate['food_amount'].apply(ast.literal_eval)
df_accommodate['testing_image_order'] = df_accommodate['testing_image_order'].apply(ast.literal_eval)
df_predict['testing_responses'] = df_predict['testing_responses'].apply(ast.literal_eval)
df_predict['food_amount'] = df_predict['food_amount'].apply(ast.literal_eval)
df_predict['testing_image_order'] = df_predict['testing_image_order'].apply(ast.literal_eval)
#Combine the dataframes and create an arbitrary column for participant numbering (the yoked orders are already stored in 'conditionOrder')
df_combined = pd.concat([df_accommodate, df_predict], ignore_index=True)
df_combined['participant'] = range(1, len(df_combined) + 1)



In [4]:
import pandas as pd


#Loop through rows and determine model parameter score for each participant

participant_rows = []

for _, row in df_combined.iterrows():
    tail_yes  = 1 if row['tail_discrete_slider.response']  == 'Yes' else 0
    wing_yes = 1 if row['wing_discrete_slider.response'] == 'Yes' else 0
    color_yes = 1 if row['color_discrete_slider.response'] == 'Yes' else 0
    model_param_score = tail_yes + wing_yes + color_yes


    participant_rows.append({
        'participant': row['participant'],
        'task': row['task'],  # predict vs accommodate
        'model_param_score': model_param_score,
        'conditionOrder': row['conditionOrder'],
        'irrelevant_dim': row['irrelevant_dim'],
        'wing_high': row['wing_high'],
        'overfit': model_param_score == 3 #overfit if all 3 dimensions selected,
    })

df_params = pd.DataFrame(participant_rows)

#Compare overfit vs not by condition
contingency = pd.crosstab(
    df_params['task'],
    df_params['overfit']
)

print(contingency)
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(contingency)

print(f"Chi-square = {chi2:.3f}")
print(f"df = {dof}")
print(f"p-value = {p:.4f}")


overfit      False  True 
task                     
accommodate    116     93
predict        121     88
Chi-square = 0.156
df = 1
p-value = 0.6930


In [5]:
df_params['intuitive_relevance'] = (
    df_params['irrelevant_dim'] != 'wing'
).astype(int)

df_params['intuitive_direction'] = (
    df_params['wing_high'] == 'T'
).astype(int)
df_params['task'] = df_params['task'].astype('category')
df_params["overfit"] = df_params["overfit"].astype(int)
print(df_params)
df_params.to_csv(os.path.join(outputdirCleaned, 'df_params_study2_for_r.csv'), index=False)

     participant         task  model_param_score  conditionOrder  \
0              1  accommodate                  3             216   
1              2  accommodate                  2              31   
2              3  accommodate                  1             142   
3              4  accommodate                  1              18   
4              5  accommodate                  3              62   
..           ...          ...                ...             ...   
413          414      predict                  2             183   
414          415      predict                  2             178   
415          416      predict                  1             172   
416          417      predict                  3               4   
417          418      predict                  1              18   

    irrelevant_dim wing_high  overfit  intuitive_relevance  \
0             wing         N        1                    0   
1             tail         N        0                    1 

In [6]:
#Map from short codes to feature descriptions

wing_map = {
    'wings': 't',
    'paws': 'n'
}

color_map = {
    'blue': 'b',
    'yellow': 'y'
}

tail_map = {
    'curly': 'c',
    'straight': 's'
}


feature_maps = {
    'wing': wing_map,
    'color': color_map,
    'tail': tail_map
}


In [7]:
#Compute feature importance scores

from doctest import debug


def compute_feature_importance_from_df(df):
    """
    Compute numeric feature importance scores(-7 to 7) for each participant,
    based on the saved slider_responses and the feature _high/_low mapping.
    This is computed based on whether a feature was really relevant (positive sign) or irrelevant (negative sign).
    0 = no response or feature was not thought to be relevant
    columns:
      - wings_discrete_slider.response, wings_direction_slider.response, wings_continuous_slider.response
      - color_discrete_slider.response, ...
      - tail_discrete_slider.response, ...
      - wings_high, wings_low, color_high, color_low, tail_high, tail_low
    """
    features = ['wing', 'color', 'tail']
    
    def compute_row_importance(row, feat):
        disc = row[f'{feat}_discrete_slider.response']
        dirc = row[f'{feat}_direction_slider.response']
        cont = row[f'{feat}_continuous_slider.response']

        #If they said a feature wasn't relevant, then importance is 0
        
        if disc == 'No' or pd.isna(disc):
            return 0.0
        
        # Make sure continuous slider value exists, if not, set it to 0
        cont_val = float(cont) if not pd.isna(cont) else 0.0

        # Get mapping from long to short feature name
        mapping = feature_maps.get(feat, {})

        # Normalize strings: strip whitespace, collapse multiple spaces, lower-case
        def normalize_str(s):
            """Strip leading/trailing whitespace and collapse internal multiple spaces."""

            if isinstance(s, str):
                return " ".join(s.split()).lower()
            return ""
        

        #Name of features need to be normalized for comparison using the mapping
        dirc_norm = normalize_str(dirc)

        #Get internal short code for selected feature direction
        internal_dirc = mapping.get(dirc_norm, None)
        
        high_val = normalize_str(row[f'{feat}_high'])
        low_val  = normalize_str(row[f'{feat}_low'])
        

        
        # Debug print statement (make sure mappings look right)
        debug = True
        if debug:
            print('response:', repr(dirc_norm), 'internal:', repr(internal_dirc), 
                'high:', repr(high_val), 'low:', repr(low_val))
            

        #If they correctly selected the high feature, assign positive sign
        if internal_dirc == high_val:
            sign = 1
        #If they incorrectly selected the low feature, assign negative sign
        elif internal_dirc == low_val:
            if debug:
                print('in negative')
            sign = -1
        else:
            if debug:
                print('in empty')
            sign = 0
            cont_val = 0.0

        # Add sign to continuous value
        importance = cont_val * sign

        return importance

    
    # Compute for each feature
    for feat in features:
        df[f'{feat}_importance'] = df.apply(lambda row: compute_row_importance(row, feat), axis=1)
    
    return df

df_combined = compute_feature_importance_from_df(df_combined)
if debug:
    print(df_combined['wing_importance'])

response: 'paws' internal: 'n' high: 'n' low: 't'
response: 'paws' internal: 'n' high: 'n' low: 't'
response: 'wings' internal: 't' high: 't' low: 'n'
response: 'paws' internal: 'n' high: 'n' low: 't'
response: 'paws' internal: 'n' high: 'n' low: 't'
response: 'wings' internal: 't' high: 't' low: 'n'
response: 'paws' internal: 'n' high: 't' low: 'n'
in negative
response: 'wings' internal: 't' high: 't' low: 'n'
response: 'wings' internal: 't' high: 't' low: 'n'
response: 'paws' internal: 'n' high: 't' low: 'n'
in negative
response: 'wings' internal: 't' high: 't' low: 'n'
response: 'paws' internal: 'n' high: 'n' low: 't'
response: 'wings' internal: 't' high: 'n' low: 't'
in negative
response: 'paws' internal: 'n' high: 't' low: 'n'
in negative
response: 'wings' internal: 't' high: 't' low: 'n'
response: 'wings' internal: 't' high: 'n' low: 't'
in negative
response: 'wings' internal: 't' high: 't' low: 'n'
response: 'wings' internal: 't' high: 't' low: 'n'
response: 'paws' internal: 'n'

In [8]:
import pandas as pd
"""Reshape to long format with 1 row per participant x feature dimension"""
# Keep only necessary columns
cols_to_keep = [
    'participant', 'task', 
    'wing_importance', 'color_importance', 'tail_importance',
    'relevant_dim_1', 'relevant_dim_2', 'irrelevant_dim', 'wing_high','color_high', 'tail_high', 'wing_discrete_slider.response'
]

df_long = df_combined[cols_to_keep].copy()

# Melt importance columns
df_long = df_long.melt(
    id_vars=['participant', 'task', 'relevant_dim_1', 'relevant_dim_2', 'irrelevant_dim', 'wing_high', 'color_high', 'tail_high', 'wing_discrete_slider.response'],
    value_vars=['wing_importance', 'color_importance', 'tail_importance'],
    var_name='feature_dimension',
    value_name='feature_importance'
)

# Simplify feature dimension names
df_long['feature_dimension'] = df_long['feature_dimension'].str.replace('_importance','')

def get_relevance(row):
    if row['feature_dimension'] in [row['relevant_dim_1'], row['relevant_dim_2']]:
        return 'relevant'
    else:
        return 'irrelevant'

df_long['feature_relevance'] = df_long.apply(get_relevance, axis=1)

print(df_long)

      participant         task relevant_dim_1 relevant_dim_2 irrelevant_dim  \
0               1  accommodate           tail          color           wing   
1               2  accommodate          color           wing           tail   
2               3  accommodate          color           tail           wing   
3               4  accommodate          color           tail           wing   
4               5  accommodate          color           tail           wing   
...           ...          ...            ...            ...            ...   
1249          414      predict          color           wing           tail   
1250          415      predict           tail          color           wing   
1251          416      predict          color           tail           wing   
1252          417      predict          color           tail           wing   
1253          418      predict          color           tail           wing   

     wing_high color_high tail_high wing_discrete_s

In [ ]:
df_overfit = df_long[df_long["feature_dimension"] == "wing"].copy()
df_overfit["wing_relevant"] = (df_overfit["feature_relevance"] == "relevant").astype(int) #Code relevancy as 0/1
df_overfit["wing_high"] = (df_overfit["wing_high"] == "T").astype(int) #Code having a wing as 0/1
df_overfit["wing_response"] = (df_overfit["wing_discrete_slider.response"] == "Yes").astype(int) #Code saying wing is relevant as 0/1
df_overfit.to_csv(os.path.join(outputdir, 'df_overfit_for_r.csv'), index=False)
df_long.to_csv(os.path.join(outputdirCleaned, 'df_long_for_R-Study2.csv'), index=False)


In [ ]:
df_wing = df_long[df_long["feature_dimension"] == "wing"].copy()
df_wing["wing_relevant"] = (df_wing["feature_relevance"] == "relevant").astype(int) #Code relevancy as 0/1
df_wing["wing_high"] = (df_wing["wing_high"] == "T").astype(int) #Code having a wing as 0/1
print(df_wing)
df_wing.to_csv(os.path.join(outputdirCleaned, 'df_wing_for_R-Study2.csv'), index=False)

     participant         task relevant_dim_1 relevant_dim_2 irrelevant_dim  \
0              1  accommodate           tail          color           wing   
1              2  accommodate          color           wing           tail   
2              3  accommodate          color           tail           wing   
3              4  accommodate          color           tail           wing   
4              5  accommodate          color           tail           wing   
..           ...          ...            ...            ...            ...   
413          414      predict          color           wing           tail   
414          415      predict           tail          color           wing   
415          416      predict          color           tail           wing   
416          417      predict          color           tail           wing   
417          418      predict          color           tail           wing   

     wing_high color_high tail_high wing_discrete_slider.respon